In [5]:
import json
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from pydantic import BaseModel

openai_client = OpenAI()

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)


Indexed 385 chunks from 95 documents


In [2]:
answer = rag('how do I implement LLM as a judge?')

In [3]:
answer

'To implement an LLM as a judge, follow these steps based on the tutorial context:\n\n1. **Installation**: Install the Evidently library with the following command:\n   ```python\n   pip install evidently\n   ```\n\n2. **Import Required Modules**: Import the necessary modules in your Python script or Jupyter Notebook:\n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset\n   from evidently import DataDefinition\n   from evidently import Report\n   from evidently import BinaryClassification\n   from evidently.descriptors import *\n   from evidently.presets import TextEvals, ValueStats, ClassificationPreset\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n3. **Set Your OpenAI API Key**: Pass your OpenAI API key as an environment variable:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n4. **Create an Evaluation Dataset**: Develop a toy Q&A dataset containing:\n   - Questions

In [4]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed


In [6]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [7]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [8]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [9]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To perform LLM evaluations, you can follow these steps:

1. **Install the Necessary Package**:
   ``
True


In [10]:
answer = rag_structured('how do I install terraform on a mac?')

print(answer.answer[:100])
print(answer.found_answer)

The provided context does not contain any information about installing Terraform on a Mac. Please ch
False


In [13]:
from typing import Optional

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool


In [12]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)


None
False


In [14]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [15]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [ ]:
answer = rag_structured('how do I use LLMs to evaluate my repo', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [18]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [19]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [20]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [21]:
answer

RAGResponse(answer='To evaluate LLMs (Large Language Models), you can utilize a panel of LLM judges to assess the generated outputs. Here’s a structured approach:\n\n## 1. Preparation\n- **Install the necessary packages:**  \n  ```python  \n  pip install evidently litellm  \n  ```  \n- **Import required components:**  \n  ```python  \n  import pandas as pd  \n  from evidently import Dataset  \n  from evidently import DataDefinition  \n  from evidently import Report  \n  from evidently.presets import TextEvals  \n  from evidently.tests import eq  \n  from evidently.descriptors import LLMEval, CustomColumnDescriptor, TestSummary  \n  ```\n\n## 2. Set up evaluator LLMs  \n- **Pass your API keys** for the LLMs you will use as judges:  \n  ```python  \n  import os  \n  os.environ["OPENAI_API_KEY"] = "YOUR KEY"  \n  os.environ["GEMINI_API_KEY"] = "YOUR KEY"  \n  os.environ["ANTHROPIC_API_KEY"] = "YOUR KEY"  \n  ```\n\n## 3. Define the data  \n- **Create a DataFrame** containing user inputs a

In [22]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [23]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [24]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not contain any information on installing Kafka on Windows.'), found_answer=False, answer=None)

In [25]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evals, you need to follow these steps:\n\n1. **Create a Project**: First, connect to Evidently Cloud and create a project using your API token:\n   ```python\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   project = ws.create_project("Your Project Name", org_id="YOUR_ORG_ID")\n   project.description = "Your project description"\n   project.save()\n   ```\n\n2. **Prepare the Dataset**: Create a dataset with questions and reference answers:\n   ```python\n   data = [\n       ["Question 1", "Answer 1"],\n       ["Question 2", "Answer 2"],\n       ...\n   ]\n   columns = ["question", "target_response"]\n   ref_data = pd.DataFrame(data, columns=columns)\n   ```\n\n3. **Create a Report and Run Eval**: Configure the evals. Use `TextEvals` to summarize output-level text evaluations:\n   ```python\n   report = Report(metrics=[TextEvals()])\n   my_eval = report.ru

In [ ]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)